In [80]:
import sqlite3
import pandas as pd

In [81]:
df_clean = pd.read_csv('../data/cleaned_retail.csv')

In [82]:
conn = sqlite3.connect(':memory:')

In [83]:
df_clean.to_sql('retail', conn, index=False, if_exists='replace')

397862

In [84]:
query_1 = """
    SELECT Country, 
           ROUND(SUM(Revenue), 2) as Total_Revenue,
           COUNT(DISTINCT CustomerID) as Unique_Customers
    FROM retail 
    GROUP BY Country 
    ORDER BY Total_Revenue DESC
    LIMIT 5
"""
print(pd.read_sql(query_1, conn))

          Country  Total_Revenue  Unique_Customers
0  United Kingdom     7039142.25              3918
1     Netherlands      285446.34                 9
2         Ireland      262171.56                 3
3         Germany      228867.14                94
4          France      199583.63                87


### Query 1: Geographic Revenue Distribution

* **Key Finding:** The analysis identifies an extreme geographic concentration risk. The United Kingdom represents the business's core market, generating $7,039,142.25 (83.7% of total revenue) and hosting 3,920 unique customers (91.1% of the total customer base).
* **International Market Metrics:** Outside the UK, the Netherlands ($285,446.34 generated by only 9 customers) and Ireland ($262,171.56 generated by 3 customers) exhibit exceptionally high Average Revenue Per User (ARPU). This indicates high-volume B2B wholesale accounts rather than standard retail shoppers.
* The business is highly vulnerable to localized economic shocks within the UK. Management should evaluate scaling infrastructure in the Netherlands and Ireland to diversify risk, leveraging the pre-existing high-value purchasing patterns in these regions.

In [85]:

query_2 = """ 
    SELECT Description, 
        ROUND(SUM(Revenue), 2) as Revenue
    FROM retail 
    GROUP BY Description 
    ORDER BY Revenue DESC
    LIMIT 20
"""
print(pd.read_sql(query_2, conn))


                           Description    Revenue
0             REGENCY CAKESTAND 3 TIER  142592.95
1   WHITE HANGING HEART T-LIGHT HOLDER  100448.15
2              JUMBO BAG RED RETROSPOT   85220.78
3                              POSTAGE   69679.21
4                        PARTY BUNTING   68844.33
5        ASSORTED COLOUR BIRD ORNAMENT   56580.34
6                   RABBIT NIGHT LIGHT   51346.20
7                        CHILLI LIGHTS   46286.51
8      PAPER CHAIN KIT 50'S CHRISTMAS    42660.83
9       PICNIC BASKET WICKER 60 PIECES   39619.50
10            BLACK RECORD COVER FRAME   39064.55
11             JUMBO BAG PINK POLKADOT   37289.59
12       DOORMAT KEEP CALM AND COME IN   35913.85
13                      SPOTTY BUNTING   35539.25
14   WOOD BLACK BOARD ANT WHITE FINISH   34478.01
15   SET OF 3 CAKE TINS PANTRY DESIGN    33347.80
16            JAM MAKING SET WITH JARS   32662.97
17                JUMBO BAG STRAWBERRY   30644.20
18     VICTORIAN GLASS HANGING T-LIGHT   28776.51


### Query 2: Product Portfolio Performance

* **Key Finding:** Revenue generation is highly dependent on a narrow group of high-performing Stock Keeping Units (SKUs). "REGENCY CAKESTAND 3 TIER" ($142,592.95) and "WHITE HANGING HEART T-LIGHT HOLDER" ($100,448.15) are now identified as the primary turnover drivers following the removal of non-retail transaction anomalies.
* **Operational Impact:** High-ranking non-product lines such as "POSTAGE" ($69,679.21) signify that shipping fees constitute a major secondary revenue stream, directly tying profitability to logistics efficiency. 
* Supply chain and procurement teams must implement strict safety stock levels for these top 20 items. A stockout in this category immediately threatens monthly net margins.

In [86]:

query_3 = """ 
    SELECT YearMonth,
      ROUND(SUM(Revenue), 2) as Monthly_Revenue 
    FROM retail 
    GROUP BY YearMonth 
    ORDER BY Monthly_Revenue DESC
"""
print(pd.read_sql(query_3, conn))

   YearMonth  Monthly_Revenue
0    2011-11       1157520.20
1    2011-10       1021772.66
2    2011-09        950805.28
3    2011-05        667967.85
4    2011-06        661213.69
5    2011-08        642843.90
6    2011-07        600091.01
7    2011-03        592126.42
8    2010-12        572713.89
9    2011-01        492261.44
10   2011-04        460507.26
11   2011-02        447137.35
12   2011-12        348141.93


### Query 3: Temporal Revenue Trends

* **Key Finding:** The financial timeline establishes a heavy Q4 cyclical seasonality. Revenue peaks sharply in October ($1,021,772.66) and November ($1,157,520.20), driven by holiday wholesale procurement. This contrast is severe compared to the Q1 baseline, where February revenue drops to an annual low of $447,137.35.
* **Operational Impact:** The business generates more revenue in October and November combined than in the entire first five months of the year (January–May).
* Working capital, warehouse staffing, and digital marketing budgets must be dynamically scaled. Capital should be preserved in Q1–Q2 to aggressively fund inventory accumulation and capacity expansion ahead of the critical Q4 volume surge.

In [87]:
unique_customers = df_clean['CustomerID'].nunique()
print(unique_customers)

4336


In [88]:
query_4 = """
    WITH customer_revenue AS (            
        SELECT CustomerID, SUM(Revenue) as Total            
        FROM retail GROUP BY CustomerID        
    ),
    ranked AS (            
        SELECT *, NTILE(10) OVER (ORDER BY Total DESC) as Decile 
        FROM customer_revenue
    )        
    SELECT Decile, ROUND(SUM(Total), 2) as Revenue_Per_Decile
    FROM ranked 
    GROUP BY Decile
    ORDER BY Decile
"""
print(pd.read_sql(query_4, conn))

   Decile  Revenue_Per_Decile
0       1          5187376.05
1       2          1173200.67
2       3           725222.35
3       4           490343.38
4       5           343683.36
5       6           252163.81
6       7           178956.93
7       8           131563.83
8       9            86801.95
9      10            45790.55


### Query 4: Customer Revenue Concentration via Decile Analysis

* **Key Finding:** The decile distribution confirms an aggressive manifestation of the Pareto Principle (the 80/20 rule). Decile 1 (the top 10% of customers by spending) accounts for $5,187,376.05, representing the vast majority of cumulative sales volume and demonstrating that business survival relies entirely on high-value, recurring buyers.
* **Risk Assessment:** The bottom 50% of customers (Deciles 6–10) contribute a negligible percentage of total financial turnover, making transactional volume from casual shoppers metrics-heavy but financially insignificant.
* The marketing focus must shift from broad customer acquisition to high-tier retention. A dedicated Key Account Management (KAM) structure and exclusive B2B loyalty workflows should be deployed immediately for Decile 1 and Decile 2 clients to mitigate churn risk.